# Scarcity-Aware Synthetic Augmentation: Google Colab GPU Execution
### Predicting When Generative Data Helps Rare Disease Classification

This notebook runs the complete experimental benchmark on **Google Colab Free Tier (T4 GPU)** with **Google Drive backup** so you never lose progress if disconnected.

#### Prerequisites:
1. Go to menu: **Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU $\rightarrow$ Save**.

In [ ]:
# 1. Verify T4 GPU Acceleration
import torch
assert torch.cuda.is_available(), "Please enable GPU in Runtime -> Change runtime type -> T4 GPU!"
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. (Optional but Recommended) Mount Google Drive for Permanent Backup
# This ensures results/runs.csv is saved to your Drive after every single experiment!
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/scarcity_paper_results"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive backup directory ready: {DRIVE_DIR}")

In [ ]:
# 3. Install Dependencies
!pip install -q medmnist scikit-learn scipy matplotlib

In [ ]:
# 4. Upload & Unpack Project Bundle or Clone from GitHub
# If using scarcity_codebase_bundle.zip uploaded to Colab files:
import os
if os.path.exists("/content/scarcity_codebase_bundle.zip"):
    !unzip -q /content/scarcity_codebase_bundle.zip -d /content/
else:
    # If cloning from GitHub, uncomment below:
    # !git clone https://github.com/<your-username>/scarcity-aware-augmentation.git /content/repo
    # %cd /content/repo
    pass

%cd /content

In [ ]:
# 5. Check if previous runs exist in Google Drive to resume automatically
import shutil
if os.path.exists(f"{DRIVE_DIR}/runs.csv"):
    os.makedirs("results", exist_ok=True)
    shutil.copy(f"{DRIVE_DIR}/runs.csv", "results/runs.csv")
    print("Restored previous runs.csv from Drive! Resume logic will skip finished experiments.")
else:
    print("Starting fresh benchmark run.")

In [ ]:
# 6. Generate Leak-Free Deterministic Splits
!python src/split.py --dataset dermamnist
!python src/split.py --dataset bloodmnist

In [ ]:
# 7. Run Full GPU Benchmark on DermaMNIST (Skin Lesions)
# 20 classifier epochs with AMP, 300 GAN epochs, K=200, 3 seeds
!python run_experiments.py --dataset dermamnist --k 200

# Sync runs.csv to Google Drive immediately
!cp results/runs.csv "$DRIVE_DIR/runs.csv"

In [ ]:
# 8. Run Full GPU Benchmark on BloodMNIST (Cross-Modality Cytology)
!python run_experiments.py --dataset bloodmnist --k 200

# Sync runs.csv to Google Drive immediately
!cp results/runs.csv "$DRIVE_DIR/runs.csv"

In [ ]:
# 9. Fit Benefit Predictor & Generate Publication Figures
!python src/predictor.py --csv results/runs.csv

# Sync all figures and tables to Drive
!cp -r results/figures/ "$DRIVE_DIR/"
!cp -r results/tables/ "$DRIVE_DIR/"

In [ ]:
# 10. Display Generated Publication Figures Inline
from IPython.display import Image, display

print("Figure 1: Rare-Class F1 vs. Scarcity (Both Modalities)")
display(Image("results/figures/fig1_gain_vs_N.png"))

print("Figure 2: Diagnostic Signals vs. Utility Gain")
display(Image("results/figures/fig2_signals_vs_gain.png"))

print("Figure 3: Depth-2 Decision Tree & Zero-Shot Transfer")
display(Image("results/figures/fig3_decision_tree_rule.png"))

In [ ]:
# 11. Download Final Paper Artifacts Bundle
!zip -r /content/final_paper_bundle.zip results/ paper/ LITERATURE_REVIEW.md
from google.colab import files
files.download("/content/final_paper_bundle.zip")